# semantitrans — idiom-aware English-speech → Hindi-text (Colab demo)

English audio in → Hindi text out, with special handling for **idiomatic** English
(e.g. *raining cats and dogs*) that plain cascades translate literally and wrong.

**Use a GPU runtime:** `Runtime → Change runtime type → Hardware accelerator → GPU`.
The code auto-detects CUDA and falls back to CPU, so it runs either way.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Clone the repo

In [ ]:
!git clone https://github.com/xonas1101/semantitrans.git
%cd semantitrans

## 3. Install dependencies

Colab already ships a CUDA build of **torch**, so we install everything *except*
torch to keep the GPU build intact.

In [ ]:
!pip install -q openai-whisper transformers sentencepiece sacremoses datasets pandas sacrebleu gTTS spacy
!python -m spacy download en_core_web_sm -q
# optional (LoRA adapter): !pip install -q peft accelerate

In [ ]:
!python verify_setup.py

## 4. Quick demo on a synthesized sentence

We synthesize an idiom-containing English sentence with TTS, then translate it
two ways: the **baseline** cascade (`--mode off`) vs the **idiom-aware** pipeline
(`--mode substitute`). First run downloads Whisper + opus-mt-en-hi (~750 MB).

In [ ]:
from gtts import gTTS
gTTS('It was raining cats and dogs last night, so we called it a day.', lang='en').save('demo.mp3')
print('wrote demo.mp3')

In [ ]:
print('=== BASELINE (no idiom handling) ===')
!python run.py demo.mp3 --mode off
print('\n=== IDIOM-AWARE (substitution) ===')
!python run.py demo.mp3 --mode substitute

## 5. Run on your own audio

Upload any English audio file (`.mp3`, `.wav`, `.m4a`, `.flac`, ...) and translate it.

In [ ]:
from google.colab import files
up = files.upload()
audio = next(iter(up))
print('uploaded:', audio)

In [ ]:
import shlex
!python run.py {shlex.quote(audio)} --mode substitute

## 6. (Optional) Build & evaluate the idiom test set

Build a MAGPIE-based test set with TTS audio, then — after filling the
`hindi_reference` column with correct Hindi — score baseline vs idiom-aware.
See the README for the full workflow.

```python
!python build_testset.py --n 60 --prefer-kb
# fill data/testset/manifest.csv -> hindi_reference, then:
!python run_testset.py --mode off --from-text
!python run_testset.py --mode substitute --from-text
!python evaluate.py
```